# `duckdb` installation and first queries

This is just a *prologue* to allow the notebooks running standalone
in a local Jupyter installation and in Google Colab.

In [ ]:
import os
file = "weatherAUS.csv.zst"
try:
    import google.colab
    IN_COLAB = True
    if not os.path.isdir("/content/oreilly-duckdb"):
        os.system("git clone https://github.com/datanizing/oreilly-duckdb")
    data = f"/content/oreilly-duckdb/data/{file}"
except:
    IN_COLAB = False
    data = f"data/{file}"

There are several ways of installing `duckdb`:
* If you want to use this Jupyter notebook, you can run `pip install duckdb pandas polars`or use the supplied
  `pyproject.toml` together with `uv`.
* If you want to run the commands in the `duckdb` CLI, you can either install it for you operating system or
  run `pip install duckdb-cli` (or `uv add duckdb-cli`).

The commands which you see here in the Jupyter notebook will also work directly in the CLI.

In [ ]:
import duckdb

You can use `duckdb` to query CSV files, even if they are compressed (like here with `zstd`, but also `gz`).
You will see later that even more formats are supported (also with plugins etc.).

Selecting data is quite easy and looks like *normal* SQL `SELECT`:

## `SELECT` and convert to `pandas` or `polars`

In [ ]:
duckdb.sql(f"SELECT * FROM '{data}'")

However, the tabular formatting is far from optimal. Fortunately, there are several options:
* If you are using a Jupyter notebook, you can convert the result to a `pandas` `DataFrame`.
* If you are working with `duckdb`'s CLI, you can use different display modes (like `.mode duckbox`, `.mode csv` etc.).

In [ ]:
duckdb.sql(f"SELECT * FROM '{data}'").df()

Be careful when converting the result to a `DataFrame`, though. One advantage of `duckdb` is that all aggregation
can take place in its engine which saves a lot of RAM. Especially for large data sets, this can not be
overestimated.

In the first `SELECT`, you can see that `duckdb` has inferred the data types. This is unfortunately not
visible in the `pandas` `DataFrame`. `polars` as an alternative to `pandas` will show the
data types of the columns. `duckdb` can also create a `polars` `DataFrame`:

In [ ]:
duckdb.sql(f"SELECT * FROM '{data}'").pl()

Different from `pandas`, `polars` shows all columns by default (this can be configured
in `pandas` though). Moreover, the data types are also visible which is very useful.

## Data Types

How does `duckdb` know about the data types and how can we find out *without*
selecting all the data? Like many other SQL databases, `duckdb` also has a
`DESCRIBE` command:

In [ ]:
duckdb.sql(f"DESCRIBE SELECT * FROM '{data}'")

`duckdb` reads the CSV file each time when we `SELECT` data. This is not
very efficient, especially for large files and if we are only interested
in aggregations or some columns.

Therefore, you normally create tables from the CSV (or JSON) files. These
tables are *not persisted*, `duckdb` (for now) works only in memory. You will
see later how the data can be made persistent, but working with temporary
tables is often very convenient:

## Temporary tables

In [ ]:
duckdb.sql(f"CREATE TABLE weather AS SELECT * FROM '{data}'")

Now, we can use `weather` as a regular SQL table (we will use the `polars` `DataFrame` from now on
to see the data types):

In [ ]:
duckdb.sql(f"SELECT * FROM weather").pl()

To save you some typing, you can omit `SELECT *`:

In [ ]:
duckdb.sql("FROM weather").pl()

In standard SQL, if you want to see only some fields, you need to explicitly
state these fields. This is useful, but often you just want to exclude one
or two fields. This is much easier in `duckdb`.
Let's exlude the fields `Sunshine` and `Evaporation`:

## `SELECT` options

In [ ]:
duckdb.sql(f"SELECT * EXCLUDE(Sunshine, Evaporation) FROM weather").pl()

If you are interested in all temperature columns, `duckdb` also offers a very
nice shortcut for choosing columns which match a certain regexp:

In [ ]:
duckdb.sql("SELECT Location, COLUMNS('.*Temp.*') FROM weather").pl()

So far, we have just used `duckdb` as an engine to read CSV files and convert
them to (typed) dataframes. This is definitely useful, but could be accomplished
with `pandas` or `polars` directly.

However, as you will see later, `duckdb` can handle much larger data volumes
and does not *spill* into the (relatively) inefficient dataframes in Python.
In typical data analysis, you will often work with *aggregates*. `duckdb`
offer super powerful aggregation functions (and much more).

Let's try a simple aggregation first:

## Aggregations

In [ ]:
duckdb.sql("SELECT Location, AVG(MinTemp), AVG(MaxTemp), AVG(Rainfall) FROM weather GROUP BY Location ORDER BY Location").pl()

Very nice. Often, you will group with respect to different columns and have
to repeat the column which you select (in this case `Location`) in the
`GROUP BY` clause. This is error-prone, if you change something.

`duckdb` makes it easier for you by supporting a `GROUP BY ALL` statement
which groups by all columns which you select and do not aggregate:

In [ ]:
duckdb.sql("SELECT Location, AVG(MinTemp), AVG(MaxTemp), AVG(Rainfall) FROM weather GROUP BY ALL ORDER BY Location").pl()

You can combine that with the pattern-based column selection:

In [ ]:
duckdb.sql("SELECT Location, AVG(COLUMNS('.*Temp.*')) FROM weather GROUP BY ALL ORDER BY Location").pl()

## Summarize

Of course, the temperatures are interesting, but that's also true for
other fields. `pandas` has a nice `df.describe()` function, and `duckdb`
offers something similar called `SUMMARIZE`:

In [ ]:
duckdb.sql("SUMMARIZE FROM weather").pl()

Inferring data types is not possible here as the different columns have different
data types.

## `SELECT` with filter

Sometimes, you want to select aggregates, but some aggregates should have different
conditions. That is no problem in standard SQL as you can use JOINs. However, `duckdb`
makes this much more comfortable by providing a `FILTER` clause in the aggregations:

In [ ]:
duckdb.sql("SELECT Location, AVG(MinTemp) as avgMin, AVG(MinTemp) FILTER (RainToday=true) AS avgMinWithRain \
            FROM weather GROUP BY ALL ORDER BY Location").pl()

Interesting, in some places the temperature seems to be higher when it's
raining. The easiest way to find those is using a subselect:

In [ ]:
duckdb.sql("FROM (SELECT Location, AVG(MinTemp) as avgMin, AVG(MinTemp) FILTER (RainToday=true) AS avgMinWithRain \
                  FROM weather GROUP BY ALL) WHERE avgMin<avgMinWithRain ORDER BY Location").pl()

With common table expressions (CTEs) this becomes more readable
without changing the results:

In [ ]:
duckdb.sql("WITH MinTempRain AS (SELECT Location, AVG(MinTemp) as avgMin, AVG(MinTemp) FILTER (RainToday=true) AS avgMinWithRain \
                                 FROM weather GROUP BY ALL)\
            SELECT * FROM MinTempRain WHERE avgMin<avgMinWithRain ORDER BY Location").pl()

## Histograms

For the continuously distributed fields, you can see the quantiles in the summary above.
If you want more details about the distribution, a histogram is often very useful.
`duckdb` can directly create these histograms which is especially nice if you do not
want to load all data into Python:

In [ ]:
duckdb.sql("SELECT Location, HISTOGRAM(COLUMNS('.*Temp.*')) FROM weather GROUP BY ALL ORDER BY Location").pl()

Here, `polars` is much better compared to `pandas`, as it understands the data structure
which `duckdb` provides. The reason is that data between those two is transmitted in the
*Apache Arrow* format.

### Alternative 1: use `duckdb` to only select one histogram

In [ ]:
duckdb.sql("WITH TempHist AS (SELECT Location, HISTOGRAM(COLUMNS('.*Temp.*')) FROM weather GROUP BY ALL), \
                 PerthMinTemp AS (SELECT unnest(map_entries(MinTemp)) AS kv FROM TempHist WHERE Location='Perth') \
            SELECT kv['key'] AS MinTemp, kv['value'] AS count FROM PerthMinTemp").pl()

In [ ]:
duckdb.sql("WITH TempHist AS (SELECT Location, HISTOGRAM(COLUMNS('.*Temp.*')) FROM weather GROUP BY ALL), \
                 PerthMinTemp AS (SELECT unnest(map_entries(MinTemp)) AS kv FROM TempHist WHERE Location='Perth') \
            SELECT kv['key'] AS MinTemp, kv['value'] AS count FROM PerthMinTemp").pl().plot.bar(x="MinTemp", y="count")

### Alternative 2: use `polars` for selection

In [ ]:
df = duckdb.sql("SELECT Location, HISTOGRAM(COLUMNS('.*Temp.*')) FROM weather GROUP BY ALL").pl()

In [ ]:
import polars as pl
# first select Perth
df.filter(pl.col("Location") == "Perth")

In [ ]:
# now use only MinTemp as a column
df.filter(pl.col("Location") == "Perth")[["MinTemp"]]

In [ ]:
# explode to convert list to separate rows
df.filter(pl.col("Location") == "Perth")[["MinTemp"]].explode("MinTemp")

In [ ]:
# unnest to create the columns
df.filter(pl.col("Location") == "Perth")[["MinTemp"]].explode("MinTemp").unnest("MinTemp")

In [ ]:
df.filter(pl.col("Location") == "Perth")[["MinTemp"]].explode("MinTemp").unnest("MinTemp").plot.bar(x="key", y="value")

## Correlations

All of the quantities above were *univariate*. Often, you might be interested
in bivariate statistics, e.g. to calculate correlations. This is easily possible
in `pandas` and `polars`, but `duckdb` can also accomplish this internally and
just return the (much shorter) data with the correlations:

In [ ]:
duckdb.sql("SELECT Location, CORR(MinTemp, MaxTemp) AS corr FROM weather GROUP BY ALL ORDER BY corr DESC").pl()

In [ ]:
import altair as alt
duckdb.sql("SELECT Location, CORR(MinTemp, MaxTemp) AS corr FROM weather GROUP BY ALL ORDER BY corr DESC").pl().\
       plot.bar(y="Location", x="corr", order=alt.Order("corr"))

Minimum and maximum temperatures are highly correlated in Cobar, only
much more weakly so in Darwin.